# AI Vocals Studio — pretrained RVC acceptance
Validates **separate first → convert isolated vocal → remix over original instrumental** using the MIT-licensed `PhoenixStormJr/RVC-V2-default-voice` test voice. No voice training required.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
ROOT=Path('/content/drive/MyDrive')
hits=list(ROOT.rglob('Kid Rock - Only God Knows Why [Official Music Video].mp3'))
assert hits, 'Source MP3 not found in MyDrive'
SOURCE=hits[0]
OUT=ROOT/'AI_Vocals_Studio'/'pretrained_rvc_test'; OUT.mkdir(parents=True,exist_ok=True)
print('Source:',SOURCE)


In [ ]:
import sys,subprocess,shutil,os
subprocess.run(['apt-get','-qq','update'],check=True)
subprocess.run(['apt-get','-qq','install','-y','ffmpeg','git'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','demucs','infer-rvc-python','huggingface_hub'],check=True)
from huggingface_hub import hf_hub_download
MODEL=hf_hub_download('PhoenixStormJr/RVC-V2-default-voice','default.pth')
INDEX=hf_hub_download('PhoenixStormJr/RVC-V2-default-voice','added_IVF511_Flat_nprobe_1_default_v2.index')
print('Downloaded pretrained test voice:',MODEL,INDEX)


In [ ]:
# Keep the acceptance render short so the free Colab test finishes quickly.
WORK=Path('/content/pretrained_rvc_acceptance'); shutil.rmtree(WORK,ignore_errors=True); WORK.mkdir(parents=True)
SRC=WORK/'source_45s.wav'
subprocess.run(['ffmpeg','-y','-ss','30','-t','45','-i',str(SOURCE),'-ar','44100','-ac','2',str(SRC)],stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL,check=True)
# FIRST AND FOREMOST: neural source separation.
SEP=WORK/'separated'
subprocess.run([sys.executable,'-m','demucs','--two-stems','vocals','-n','htdemucs','-o',str(SEP),str(SRC)],check=True)
VOCALS=next(SEP.rglob('vocals.wav')); BEAT=next(SEP.rglob('no_vocals.wav'))
assert VOCALS.stat().st_size>10000 and BEAT.stat().st_size>10000
print('Separated vocals:',VOCALS); print('Original separated instrumental:',BEAT)


In [ ]:
# Convert ONLY the isolated vocal with the already-trained RVC voice.
from infer_rvc_python import BaseLoader
loader=BaseLoader(only_cpu=False)
loader.apply_conf(tag='acceptance',file_model=MODEL,pitch_algo='rmvpe+',pitch_lvl=0,file_index=INDEX,index_influence=0.75,envelope_ratio=0.25,consonant_breath_protection=0.5)
converted=loader([str(VOCALS)],'acceptance')[0]
CONVERTED=Path(converted); assert CONVERTED.exists() and CONVERTED.stat().st_size>10000
print('Converted isolated vocal:',CONVERTED)


In [ ]:
# Remix converted vocal over the untouched separated instrumental.
FINAL_WAV=WORK/'pretrained_rvc_remix.wav'; FINAL_MP3=OUT/'pretrained_rvc_remix_45s.mp3'
subprocess.run(['ffmpeg','-y','-i',str(BEAT),'-i',str(CONVERTED),'-filter_complex','[0:a][1:a]amix=inputs=2:duration=first:dropout_transition=0,alimiter=limit=0.98[a]','-map','[a]','-ar','44100',str(FINAL_WAV)],check=True)
subprocess.run(['ffmpeg','-y','-i',str(FINAL_WAV),'-codec:a','libmp3lame','-b:a','192k',str(FINAL_MP3)],check=True)
assert FINAL_MP3.exists() and FINAL_MP3.stat().st_size>100000
print('PASS: separate -> pretrained RVC conversion -> original-beat remix')
print('Saved:',FINAL_MP3)
from IPython.display import Audio,display
display(Audio(str(FINAL_MP3)))
